# SFSC 2026 — V5 Step 0: free gains and an honest baseline (CPU only, no training)

Step 0 of `v5/V5_PLAN.md`. It reads the finished V3 and V4 runs (read only) and tests cheap changes to the V4 system:

| Experiment | What changes | Why |
|---|---|---|
| **recall probe** | candidates from the UNet at `t_low` 0.3 / 0.2, prefilter off, scorer refitted | `t_low = 0.4` is the binding gate; a lower one adds recall the scorer was never allowed to price |
| **Otsu re-snap** | every UNet candidate is re-drawn on the dark H-alpha pixels around it (`snap` replaces, `grow` only adds) | 813 near-misses (IoU 0.1-0.5) are mostly outline errors |
| **combination** | best recall variant + best re-snap variant, if both pass | |
| **nested-selection audit** | variant and threshold chosen on 2 folds, scored on the other 2 | how optimistic our out-of-fold numbers are |
| **exports** | best system, the same with boundary offset 0, the same with a loose threshold (~12 kept per image) | 3 leaderboard probes; no test image is ever left empty |

Every number is **cross-fitted** on V3's folds. A variant is adopted only if it beats the V4 replica by `min_gain` **and** wins in
`min_fold_wins` of 4 folds. The `base` variant must reproduce V4's out-of-fold PQ (0.4550).

**Run:** CPU runtime with high RAM is enough. `Runtime -> Run all`. Outputs go to `MyDrive/SFSC2026/outputs/sfsc_v5/step0`.
Drive access: `drive.mount` in the Colab UI; set `SFSC_DRIVE_MODE=rclone` for headless runs (needs an rclone config with a
Google Drive remote, name in `SFSC_RCLONE_REMOTE`).


## Cell 01 — Drive access, input runs (read only), output folder


In [ ]:
from pathlib import Path
import os, shutil, subprocess

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_MODE = os.environ.get("SFSC_DRIVE_MODE", "mount" if IN_COLAB else "local")   # mount | rclone | local
RCLONE_REMOTE = os.environ.get("SFSC_RCLONE_REMOTE", "gdrive")
LOCAL_ROOT = Path(os.environ.get("SFSC_LOCAL_ROOT", "/content/sfsc_v5" if IN_COLAB else "/tmp/sfsc_v5"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA = LOCAL_ROOT / "dataset"

def rclone(*args):
    cmd = ["rclone", *args, "--transfers", "16", "--checkers", "16", "--drive-chunk-size", "64M"]
    subprocess.run(cmd, check=True)

if DRIVE_MODE == "mount":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SFSC2026")
elif DRIVE_MODE == "rclone":
    if shutil.which("rclone") is None:
        subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True, check=True)
    DRIVE_ROOT = LOCAL_ROOT / "drive_mirror"
    remote = f"{RCLONE_REMOTE}:SFSC2026"
    rclone("copy", f"{remote}/data", str(DRIVE_ROOT / "data"), "--include", "/sfsc2026-data.zip")
    rclone("copy", f"{remote}/outputs/sfsc_unet_v3", str(DRIVE_ROOT / "outputs" / "sfsc_unet_v3"),
           "--include", "/*.{csv,json,npz}", "--include", "/fold*/{config,done}.json",
           "--include", "/oof/**", "--include", "/test_probs/**")
    rclone("copy", f"{remote}/outputs/sfsc_fusion_v4", str(DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"),
           "--include", "/*.json", "--include", "/detections/**", "--include", "/detector/fold*/done.json")
    if os.environ.get("SFSC_DET_RUN"):
        rclone("copy", f"{remote}/outputs/{os.environ['SFSC_DET_RUN']}", str(DRIVE_ROOT / "outputs" / os.environ["SFSC_DET_RUN"]),
               "--include", "/detections/**", "--include", "/detector/fold*/done.json")
else:
    DRIVE_ROOT = Path(os.environ["SFSC_DRIVE_ROOT"])

DATA_ZIP = DRIVE_ROOT / "data" / "sfsc2026-data.zip"
DRIVE_EXTRACTED = DRIVE_ROOT / "data" / "SFSCDataset"
V3_RUN = DRIVE_ROOT / "outputs" / "sfsc_unet_v3"            # read only
V4_RUN = DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"          # read only
DET_RUN_NAME = os.environ.get("SFSC_DET_RUN", "sfsc_fusion_v4")      # e.g. sfsc_v5/seeds for the V5 detector
DET_RUN = DRIVE_ROOT / "outputs" / DET_RUN_NAME
OUT_BASE = DRIVE_ROOT / "outputs" / "sfsc_v5" / os.environ.get("SFSC_OUT_NAME", "step0")

for _needed in (V3_RUN / "folds_v3.csv", V3_RUN / "frozen_recipe_v3.json", V4_RUN / "manifest_v4.json"):
    if not _needed.is_file():
        raise FileNotFoundError(f"{_needed} not found. Step 0 needs the finished V3 and V4 runs.")

def push_outputs():
    """Mount mode writes straight to Drive; rclone mode copies the output folder back."""
    if DRIVE_MODE == "rclone":
        rclone("copy", str(RUN), f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{RUN.name}")

print("Drive mode:", DRIVE_MODE, "| V3:", V3_RUN, "| V4:", V4_RUN, "| output base:", OUT_BASE)


## Cell 02 — Dependencies (CPU only; no PyTorch or Ultralytics needed)


In [ ]:
import importlib.util, sys

REQUIRED = {"sklearn": "scikit-learn>=1.3", "pycocotools": "pycocotools>=2.0.7", "cv2": "opencv-python-headless>=4.8",
            "scipy": "scipy>=1.10", "pandas": "pandas>=2.0", "tqdm": "tqdm>=4.66"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import sklearn
print("sklearn", sklearn.__version__)


## Cell 03 — Imports, configuration (edit here), settings inherited from V3 and V4, helpers


In [ ]:
import copy, gc, hashlib, json, math, multiprocessing as mp, pickle, platform
import random, re, shutil, subprocess, sys, tempfile, time, warnings, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from dataclasses import dataclass, asdict
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
from sklearn.ensemble import HistGradientBoostingRegressor
from tqdm.auto import tqdm


@dataclass
class Config:
    # ---- run control ----
    debug_run: bool = os.environ.get("SFSC_DEBUG", "0") == "1"   # True = quick check on 24 images per fold
    local_dataset_override: str | None = os.environ.get("SFSC_DATASET") or None
    export_test: bool = True
    skip_empty_records: bool = True          # the organizers' scorer never sees annotator sets without filaments

    # ---- adoption rule ----
    min_gain: float = 0.005                  # out-of-fold PQ a variant must gain over the V4 replica...
    min_fold_wins: int = 3                   # ...while winning in at least this many folds
    bootstrap: int = 2000
    v4_reference_pq: float = 0.45501775638338393   # V4 manifest, adopted v4_full; the base variant must reproduce it

    # ---- recall probe ----
    probe_t_lows: tuple = (0.3, 0.2)
    probe_prefilter_area: int = 16           # with the prefilter off, components below this area are still dropped

    # ---- Otsu re-snap ----
    resnap_margin: int = 16                  # px around the candidate's box used for the local Otsu split
    resnap_grow: int = 3                     # dark pixels may extend this many px beyond the candidate
    resnap_close: int = 3                    # closing kernel that bridges 1-px gaps
    resnap_area_lo: float = 0.5              # fall back to the original mask if the area changes outside [lo, hi]x
    resnap_area_hi: float = 2.0
    resnap_min_pixels: int = 64              # on-disk pixels needed in the window for an Otsu split

    # ---- exports ----
    loose_keep_per_image: float = 12.0       # leaderboard probe: threshold that keeps this many instances per image
    strict_keep_per_image: tuple = (4.0, 3.0)  # stricter leaderboard probes (V4 keeps ~6 per image)
    never_empty: bool = True                 # an image with no kept candidate keeps its best-scored one
    cpu_pool_workers: int | None = int(os.environ['SFSC_CPU_POOL']) if os.environ.get('SFSC_CPU_POOL') else None

CFG = Config()
DEBUG_LIMIT = 24 if CFG.debug_run else None

# ---- settings that must match V3 and V4 are read from those runs, not retyped ----
_cfg_files = sorted(V3_RUN.glob("fold*/config.json"))
if not _cfg_files:
    raise FileNotFoundError(f"No fold*/config.json under {V3_RUN}")
V3CFG = json.loads(_cfg_files[0].read_text())["config"]
for _k in ("seed", "n_folds", "disk_radius_fraction", "radial_bins", "flat_median_kernel", "flat_std", "contact_radius",
           "connectivity", "fill_holes", "min_core_area", "contact_assign_radius", "scorer_prefilter_max",
           "scorer_prefilter_area"):
    setattr(CFG, _k, V3CFG[_k])
CFG.expected_shape = tuple(int(x) for x in V3CFG["expected_shape"])
CFG.scorer_taus = tuple(float(t) for t in V3CFG["scorer_taus"])
V4_MANIFEST = json.loads((V4_RUN / "manifest_v4.json").read_text())
V4CFG = V4_MANIFEST["config"]
for _k in ("det_min_area", "det_link_frac", "det_only_floor", "det_only_max_claimed", "det_grow", "det_claim_t",
           "det_claim_min_frac", "det_only_min_area"):
    setattr(CFG, _k, V4CFG[_k])
if V4_MANIFEST["adopted"] != "v4_full":
    warnings.warn(f"V4 adopted {V4_MANIFEST['adopted']}, not v4_full; the base variant replicates v4_full.")

RUN = OUT_BASE.with_name(OUT_BASE.name + ("_debug" if CFG.debug_run else ""))
RUN.mkdir(parents=True, exist_ok=True)
random.seed(CFG.seed); np.random.seed(CFG.seed)
cv2.setNumThreads(0)

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
CPU_POOL = max(2, CPU_COUNT - 1) if CFG.cpu_pool_workers is None else CFG.cpu_pool_workers
T0 = time.time()

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".json_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2, default=str)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def save_npy_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".npy_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as f:
            np.save(f, value, allow_pickle=False)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def copy_atomic(src, dst):
    """Copy unless a file of the same size is already there."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        return
    copy_force(src, dst)

def copy_force(src, dst):
    """Always copy (checkpoints of the same size differ in content)."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_suffix(dst.suffix + ".part")
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def save_pickle(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    with open(temp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, path)


print("CPU cores:", CPU_COUNT, "| CPU scoring pool:", CPU_POOL)
print("Run folder:", RUN, "| debug_run =", CFG.debug_run)
print("From V3: seed", CFG.seed, "| prefilter max", CFG.scorer_prefilter_max, "area", CFG.scorer_prefilter_area,
      "| taus", CFG.scorer_taus[0], "...", CFG.scorer_taus[-1])


## Cell 04 — Stage the dataset on the local SSD (unchanged from V4)


In [ ]:
ANNOTATION_BASENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
LOCAL_ZIP = LOCAL_ROOT / "sfsc_dataset.zip"

def locate_dataset(root):
    root = Path(root)
    hits = sorted(root.rglob(ANNOTATION_BASENAME)) if root.is_dir() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {ANNOTATION_BASENAME} under {root}; found {len(hits)}")
    ann = hits[0]
    train_dir = ann.parent / "train_images"
    test_dir = ann.parent.parent / "test" / "test_images"
    if not (train_dir.is_dir() and test_dir.is_dir()):
        raise FileNotFoundError(f"Missing train_images or test_images near {ann}")
    return ann, train_dir, test_dir

def image_paths(folder):
    return {p.name: p for p in Path(folder).iterdir()
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")}

def staged_and_ready(root):
    try:
        a, t, s = locate_dataset(root)
        return a.stat().st_size > 1000 and len(image_paths(t)) > 0 and len(image_paths(s)) > 0
    except (FileNotFoundError, OSError):
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc="Extract dataset"):
            if member.is_dir():
                continue
            dest = (destination / member.filename).resolve()
            if not dest.is_relative_to(resolved_root):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
            if dest.is_file() and dest.stat().st_size == member.file_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            archive.extract(member, destination)

if CFG.local_dataset_override:
    DATA_ROOT = Path(CFG.local_dataset_override)
    if not staged_and_ready(DATA_ROOT):
        raise FileNotFoundError(f"Dataset override not ready: {DATA_ROOT}")
elif staged_and_ready(LOCAL_DATA):
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = LOCAL_DATA
    if DATA_ZIP.is_file():
        print("Copying ZIP to local SSD once per runtime...")
        copy_atomic(DATA_ZIP, LOCAL_ZIP)
        safe_extract(LOCAL_ZIP, DATA_ROOT)
    elif DRIVE_EXTRACTED.is_dir():
        print("Copying extracted dataset folder from Drive to local SSD...")
        shutil.copytree(DRIVE_EXTRACTED, DATA_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"Expected {DATA_ZIP} or {DRIVE_EXTRACTED}. Or set CFG.local_dataset_override.")
if not staged_and_ready(DATA_ROOT):
    raise RuntimeError("Local staging incomplete. Check ZIP contents and free disk space.")

ANN_PATH, TRAIN_DIR, TEST_DIR = locate_dataset(DATA_ROOT)
TRAIN_PATHS, TEST_PATHS = image_paths(TRAIN_DIR), image_paths(TEST_DIR)
for label, paths in (("train", TRAIN_PATHS), ("test", TEST_PATHS)):
    if len({Path(n).stem for n in paths}) != len(paths):
        raise ValueError(f"Duplicate {label} image stems; IDs would collide.")

def read_gray(path):
    with Image.open(path) as im:
        if im.mode == "L":
            return np.asarray(im, dtype=np.uint8).copy()
        if im.mode != "RGB":
            raise ValueError(f"Unexpected image mode {im.mode}: {path}")
        arr = np.asarray(im, dtype=np.uint8)
        d1 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 1])
        d2 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 2])
        if d1.max() > 2 or d2.max() > 2:
            raise ValueError(f"Non-grayscale RGB image: {path}")
        return arr[:, :, 0].copy()

RAW = json.loads(ANN_PATH.read_text())
ANN_SHA = hash_file(ANN_PATH)
print("Annotation JSON:", ANN_PATH, "| sha256", ANN_SHA[:16])
print("JPEGs:", len(TRAIN_PATHS), "train /", len(TEST_PATHS), "test | free local GiB:",
      round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 1))


## Cell 05 — COCO parsing, record index (unchanged from V4)


In [ ]:
H, W = CFG.expected_shape

def native_rle(rle):
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    return {"size": [int(x) for x in rle["size"]], "counts": counts}

def json_rle(rle):
    counts = rle["counts"]
    return {"size": [int(x) for x in rle["size"]],
            "counts": counts.decode("ascii") if isinstance(counts, bytes) else counts}

def decode_mask(rle):
    a = mask_api.decode(native_rle(rle))
    return np.any(a, axis=2) if a.ndim == 3 else (a > 0)

def segmentation_to_rle(seg, h, w):
    if isinstance(seg, dict):
        if list(seg.get("size", [h, w])) != [h, w]:
            raise ValueError("Annotation RLE shape mismatch")
        rle = (mask_api.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list)
               else native_rle({"size": [h, w], "counts": seg["counts"]}))
    elif isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        if any(len(p) < 6 or len(p) % 2 for p in polys):
            raise ValueError("Malformed polygon segmentation")
        rle = mask_api.merge(mask_api.frPyObjects(polys, h, w))
    else:
        raise ValueError("Unsupported empty/malformed segmentation")
    return json_rle(rle)

STEMS = defaultdict(list)
for _n in TRAIN_PATHS:
    STEMS[Path(_n).stem].append(_n)

def canonical_filename(s):
    name = Path(str(s)).name
    if name in TRAIN_PATHS:
        return name
    cands = STEMS.get(Path(name).stem, [])
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"Unresolvable annotated JPEG: {s}")

IMAGES_BY_ID = {}
for im in RAW["images"]:
    rid = str(im["id"])
    if rid in IMAGES_BY_ID:
        raise ValueError(f"Duplicate image record {rid}")
    if (int(im["height"]), int(im["width"])) != (H, W):
        raise ValueError(f"Unexpected dimensions for record {rid}")
    IMAGES_BY_ID[rid] = im

ANNOTATIONS_BY_ID = defaultdict(list)
_seen = set()
for a in RAW["annotations"]:
    aid, rid = str(a["id"]), str(a["image_id"])
    if aid in _seen or rid not in IMAGES_BY_ID:
        raise ValueError(f"Duplicate annotation or missing image: {aid} / {rid}")
    if int(a.get("iscrowd", 0)):
        raise ValueError(f"Unexpected crowd annotation {aid}")
    if int(a.get("category_id", 1)) not in (1, 2, 3, 4):
        raise ValueError(f"Unexpected category {a.get('category_id')} in {aid}")
    _seen.add(aid)
    ANNOTATIONS_BY_ID[rid].append(a)

RECORD_IDS_BY_FILE = defaultdict(list)
for rid, im in IMAGES_BY_ID.items():
    RECORD_IDS_BY_FILE[canonical_filename(im["file_name"])].append(rid)
for k in RECORD_IDS_BY_FILE:
    RECORD_IDS_BY_FILE[k].sort()
UNIQUE_FILES = sorted(RECORD_IDS_BY_FILE)

FILE_HASHES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    for name, digest in zip(UNIQUE_FILES, pool.map(lambda n: hash_file(TRAIN_PATHS[n]), UNIQUE_FILES)):
        FILE_HASHES[name] = digest

unannotated = sorted(set(TRAIN_PATHS) - set(UNIQUE_FILES))
if unannotated:
    warnings.warn(f"{len(unannotated)} train JPEGs have no annotation record; they are not used.")
print("Annotated JPEGs:", len(UNIQUE_FILES), "| annotator records:", len(IMAGES_BY_ID),
      "| instances:", len(RAW["annotations"]))
print("Records per JPEG:", pd.Series([len(v) for v in RECORD_IDS_BY_FILE.values()]).value_counts().sort_index().to_dict())


## Cell 06 — Disk geometry, flattening, image caches, ground-truth index (unchanged from V4)


In [ ]:
CY, CX = (H - 1) / 2.0, (W - 1) / 2.0
DISK_R = CFG.disk_radius_fraction * min(H, W) / 2.0
_YY, _XX = np.mgrid[0:H, 0:W]
RADIAL = (np.hypot(_YY - CY, _XX - CX) / DISK_R).astype(np.float32)
YY_FLAT = _YY.ravel().astype(np.float64)
XX_FLAT = _XX.ravel().astype(np.float64)
del _YY, _XX
DISK = RADIAL < 1.0
RBIN = np.minimum((RADIAL * CFG.radial_bins).astype(np.int32), CFG.radial_bins - 1)
RBIN_DISK = RBIN[DISK]

IMG_MEAN, IMG_STD = 0.5, 0.25
RAD_MEAN, RAD_STD, RAD_CLIP = 0.6, 0.35, 1.25

def flatten_disk(img):
    """Divide by the radial limb-darkening profile, then by a large local median; off-disk = neutral (1.0 -> 128)."""
    x = img.astype(np.float32)
    nb = CFG.radial_bins
    s = np.bincount(RBIN_DISK, weights=x[DISK], minlength=nb)
    c = np.bincount(RBIN_DISK, minlength=nb)
    valid = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(valid), (s[valid] / c[valid]))
    k = 7
    prof = np.convolve(np.pad(prof, k // 2, mode="edge"), np.ones(k) / k, mode="valid")
    ratio = x / np.maximum(prof[RBIN], 1.0).astype(np.float32)
    ratio[~DISK] = 1.0
    small = cv2.resize(ratio, (W // 4, H // 4), interpolation=cv2.INTER_AREA)
    small_u8 = np.clip(small * 128.0, 0, 255).astype(np.uint8)
    ksize = max(3, int(round(CFG.flat_median_kernel * H / 2048)) | 1)
    med = cv2.medianBlur(small_u8, ksize).astype(np.float32) / 128.0
    med = cv2.resize(med, (W, H), interpolation=cv2.INTER_LINEAR)
    flat = ratio / np.maximum(med, 0.25)
    flat[~DISK] = 1.0
    return np.clip(flat * 128.0, 0, 255).astype(np.uint8)

CACHE = LOCAL_ROOT / "cache" / f"v3_{ANN_SHA[:12]}_{H}x{W}_r{CFG.disk_radius_fraction:.4f}_k{CFG.flat_median_kernel}_c{CFG.contact_radius}"
for sub in ("img", "flat", "soft", "contact", "gt"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)

def cache_key(split, name):
    return f"{split}__{Path(name).stem}"

def cache_image(split, name, path):
    key = cache_key(split, name)
    ip, fp = CACHE / "img" / f"{key}.npy", CACHE / "flat" / f"{key}.npy"
    if ip.is_file() and fp.is_file():
        return
    img = read_gray(path)
    if img.shape != (H, W):
        raise ValueError(f"Wrong JPEG shape {img.shape}: {path}")
    save_npy_atomic(ip, img)
    save_npy_atomic(fp, flatten_disk(img))
def build_gt(name):
    """Flat pixel indices and areas of every annotated filament, per annotator set."""
    gp = CACHE / "gt" / f"{cache_key('train', name)}_v4.pkl"
    if gp.is_file():
        with open(gp, "rb") as f:
            return pickle.load(f)
    records = []
    for rid in RECORD_IDS_BY_FILE[name]:
        idx_list, areas = [], []
        for ann in ANNOTATIONS_BY_ID[rid]:
            m = decode_mask(segmentation_to_rle(ann["segmentation"], H, W))
            if not m.any():
                raise ValueError(f"Empty filament annotation {ann['id']}")
            flat_idx = np.flatnonzero(m).astype(np.int32)
            idx_list.append(flat_idx); areas.append(len(flat_idx))
        records.append({"id": rid, "idx": idx_list, "area": np.asarray(areas, np.float64)})
    gt = {"name": name, "records": records}
    save_pickle(gp, gt)
    return gt

with ThreadPoolExecutor(max_workers=min(8, CPU_COUNT)) as pool:
    list(tqdm(pool.map(lambda n: cache_image("train", n, TRAIN_PATHS[n]), UNIQUE_FILES),
              total=len(UNIQUE_FILES), desc="Cache train images + flattened"))
    list(tqdm(pool.map(lambda n: cache_image("test", n, TEST_PATHS[n]), sorted(TEST_PATHS)),
              total=len(TEST_PATHS), desc="Cache test images + flattened"))
    _gt_list = list(tqdm(pool.map(build_gt, UNIQUE_FILES), total=len(UNIQUE_FILES), desc="Ground-truth index"))

GT_INDEX, N_EMPTY_RECORDS = {}, 0
for g in _gt_list:
    scored = [r for r in g["records"] if len(r["idx"])] if CFG.skip_empty_records else list(g["records"])
    N_EMPTY_RECORDS += len(g["records"]) - len(scored)
    GT_INDEX[g["name"]] = {"name": g["name"], "records": scored, "n_all": len(g["records"])}
del _gt_list

@lru_cache(maxsize=64)
def cached_arrays(key, with_targets=True):
    """(image, flattened image) as memory maps. The second argument is kept for compatibility with V3's functions."""
    return tuple(np.load(CACHE / s / f"{key}.npy", mmap_mode="r") for s in ("img", "flat"))

print("Scored annotator sets:", sum(len(g["records"]) for g in GT_INDEX.values()),
      "| filaments:", sum(len(r["idx"]) for g in GT_INDEX.values() for r in g["records"]),
      "| annotator sets without filaments (skipped: %s): %d" % (CFG.skip_empty_records, N_EMPTY_RECORDS))


## Cell 07 — V3 folds, recipe, out-of-fold maps and test maps (unchanged from V4)


In [ ]:
FOLDS = pd.read_csv(V3_RUN / "folds_v3.csv", dtype={"file_name": str, "group": str, "v1_split": str})
if set(FOLDS.file_name) != set(UNIQUE_FILES):
    raise ValueError("V3's folds_v3.csv does not match this dataset.")
FOLDS["fold"] = FOLDS["fold"].astype(int)
FOLD_OF = dict(zip(FOLDS.file_name, FOLDS.fold))

V3_FROZEN = json.loads((V3_RUN / "frozen_recipe_v3.json").read_text())
V3_REC, V3_SHA8 = V3_FROZEN["recipe"], V3_FROZEN["sha256"][:8]
_g = V3_REC["geometry"]
GEOM = (float(_g["t_low"]), None if _g["contact"] is None else float(_g["contact"]), int(_g["offset"]))
V3_TAU = None if V3_REC.get("scorer_tau") is None else float(V3_REC["scorer_tau"])
V3_FOLDS = sorted(int(f) for f in V3_REC["folds"])
OOF_TAG = object_hash({"tta": list(V3CFG["tta_final"]), "tile": V3CFG["tile"], "stride": V3CFG["stride"]})[:10]
MAPS_LOCAL = LOCAL_ROOT / "v3_maps"

def stage_npz(task):
    """Drive .npz (fg, ct) -> two local .npy files that CPU workers can memory-map."""
    src, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        with np.load(src) as z:
            fg, ct = z["fg"], z["ct"]
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def find_oof_dir(fold):
    sha = json.loads((V3_RUN / f"fold{fold}" / "done.json").read_text())["best_sha256"][:10]
    exact = V3_RUN / "oof" / f"fold{fold}_{sha}_{OOF_TAG}"
    if exact.is_dir():
        return exact
    cands = sorted((d for d in (V3_RUN / "oof").glob(f"fold{fold}_*") if d.is_dir()),
                   key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {V3_RUN / 'oof'}. Run V3's Cell 19.")
    return cands[-1]

_tasks, _owners = [], []
for fold in V3_FOLDS:
    d = find_oof_dir(fold)
    names = sorted(FOLDS.loc[FOLDS.fold == fold, "file_name"])
    if DEBUG_LIMIT:
        names = names[:DEBUG_LIMIT]
    for name in names:
        stem = Path(name).stem
        src = d / f"{stem}.npz"
        if not src.is_file():
            raise FileNotFoundError(f"Missing V3 out-of-fold map {src}. Run V3's Cell 19 to completion.")
        _tasks.append((src, MAPS_LOCAL / d.name / f"{stem}_fg.npy", MAPS_LOCAL / d.name / f"{stem}_ct.npy"))
        _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    OOF_PATHS = dict(zip(_owners, tqdm(pool.map(stage_npz, _tasks), total=len(_tasks), desc="Stage V3 out-of-fold maps")))

_tasks, _owners = [], []
for name in sorted(TEST_PATHS)[:DEBUG_LIMIT] if DEBUG_LIMIT else sorted(TEST_PATHS):
    stem = Path(name).stem
    src = V3_RUN / "test_probs" / f"{stem}_{V3_SHA8}.npz"
    if not src.is_file():
        raise FileNotFoundError(f"Missing V3 test map {src}. Run V3's Cell 23 to completion.")
    _tasks.append((src, MAPS_LOCAL / f"test_{V3_SHA8}" / f"{stem}_fg.npy", MAPS_LOCAL / f"test_{V3_SHA8}" / f"{stem}_ct.npy"))
    _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    TEST_MAPS = dict(zip(_owners, tqdm(pool.map(stage_npz, _tasks), total=len(_tasks), desc="Stage V3 test maps")))

print("V3 geometry:", GEOM, "| V3 keep threshold:", V3_TAU, "| V3 cross-fitted out-of-fold PQ:", V3_REC.get("oof_pq_scorer"))
print("V3 maps staged:", len(OOF_PATHS), "out-of-fold,", len(TEST_MAPS), "test")


## Cell 08 — Exact PQ engine and instance extraction (unchanged from V4)


In [ ]:
STRUCT8 = ndi.generate_binary_structure(2, 2)
STRUCT4 = ndi.generate_binary_structure(2, 1)
CONN = STRUCT8 if CFG.connectivity == 8 else STRUCT4
FEATURES = ("log_area", "mean_p", "max_p", "frac50", "frac70", "frac90", "mean_contact",
            "elongation", "major_len", "radial", "mean_img", "mean_flat")
F_IDX = {k: i for i, k in enumerate(FEATURES)}

def qt(t):
    return int(round(float(t) * 255))

def match_stats(iou):
    """Strict IoU>0.5, one-to-one max-IoU assignment. Returns (sum_iou, tp, fp, fn, records)."""
    n_gt, n_pred = iou.shape
    if n_gt == 0 or n_pred == 0:
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    q = iou > 0.5
    if not q.any():
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    rows, cols = np.flatnonzero(q.any(1)), np.flatnonzero(q.any(0))
    sub = np.where(q[np.ix_(rows, cols)], iou[np.ix_(rows, cols)], 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    v = sub[r, c]
    v = v[v > 0]
    tp = len(v)
    return np.array([float(v.sum()), tp, n_pred - tp, n_gt - tp, 1], np.float64)

def match_pairs(iou):
    if iou.size == 0 or not (iou > 0.5).any():
        return []
    sub = np.where(iou > 0.5, iou, 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    return [(int(a), int(b), float(sub[a, b])) for a, b in zip(r, c) if sub[a, b] > 0]

def summarize(st):
    s, tp, fp, fn, rec = [float(x) for x in st]
    den = tp + 0.5 * (fp + fn)
    return {"pq": s / den if den else 1.0, "sq": s / tp if tp else 0.0, "rq": tp / den if den else 1.0,
            "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / (tp + fn) if tp + fn else 0.0,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "records": int(rec)}

def relabel(lab):
    if lab.max() == 0:
        return np.zeros(lab.shape, np.int32), 0
    areas = np.bincount(lab.ravel())
    ids = np.flatnonzero(areas[1:]) + 1
    lut = np.zeros(len(areas), np.int32)
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    return lut[lab], len(ids)

def fill_label_holes(lab):
    fg = lab > 0
    holes = ndi.binary_fill_holes(fg) & ~fg
    if not holes.any():
        return lab
    hl, _ = ndi.label(holes, structure=STRUCT4)
    out = lab.copy()
    for i, sl in enumerate(ndi.find_objects(hl), 1):
        if sl is None:
            continue
        esl = (slice(max(sl[0].start - 1, 0), sl[0].stop + 1), slice(max(sl[1].start - 1, 0), sl[1].stop + 1))
        hole = hl[esl] == i
        ring = ndi.binary_dilation(hole, STRUCT8) & ~hole
        vals = lab[esl][ring]
        vals = vals[vals > 0]
        if vals.size:
            out[esl][hole] = np.bincount(vals).argmax()
    return out

def geometry_labels(fg, ct, geom):
    t_low, contact_t, offset = geom
    low = fg >= qt(t_low)
    if contact_t is not None:
        cores = low & (ct < qt(contact_t))
        lab, n = ndi.label(cores, structure=CONN)
        if n:
            areas = np.bincount(lab.ravel(), minlength=n + 1)
            small = areas < CFG.min_core_area
            small[0] = False
            if small.any():
                lab[small[lab]] = 0
        if lab.any():
            dist, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_distances=True, return_indices=True)
            lab = np.where(low & (dist <= CFG.contact_assign_radius), lab[iy, ix], 0).astype(np.int32)
        else:
            lab = np.zeros((fg.shape), np.int32)
    else:
        lab, _ = ndi.label(low, structure=CONN)
        lab = lab.astype(np.int32)
    if CFG.fill_holes and lab.any():
        lab = fill_label_holes(lab)
    for _ in range(max(offset, 0)):
        lab = np.where(lab == 0, ndi.grey_dilation(lab, footprint=STRUCT4), lab)
    for _ in range(max(-offset, 0)):
        lab = np.where(ndi.grey_erosion(lab, footprint=STRUCT4) == lab, lab, 0)
    return relabel(lab)

def component_features(lab, n, fg, ct, img, flat):
    if n == 0:
        return np.zeros((0, len(FEATURES)), np.float32), np.zeros(0, np.float64)
    L = lab.ravel()
    nz = np.flatnonzero(L)
    Lz = L[nz]
    def s(w):
        return np.bincount(Lz, weights=w, minlength=n + 1)[1:]
    area = np.bincount(Lz, minlength=n + 1)[1:].astype(np.float64)
    a = np.maximum(area, 1.0)
    fgz = np.asarray(fg).ravel()[nz]
    p = fgz.astype(np.float64) / 255.0
    mx = np.zeros(n + 1, np.float64)
    np.maximum.at(mx, Lz, fgz.astype(np.float64))
    yz, xz = YY_FLAT[nz], XX_FLAT[nz]
    my, mxx = s(yz) / a, s(xz) / a
    syy = np.maximum(s(yz * yz) / a - my ** 2, 0)
    sxx = np.maximum(s(xz * xz) / a - mxx ** 2, 0)
    sxy = s(yz * xz) / a - my * mxx
    tr, det = syy + sxx, syy * sxx - sxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 1e-3)
    feats = np.stack([
        np.log1p(area), s(p) / a, mx[1:] / 255.0,
        s((fgz >= qt(0.5)).astype(np.float64)) / a, s((fgz >= qt(0.7)).astype(np.float64)) / a,
        s((fgz >= qt(0.9)).astype(np.float64)) / a,
        s(np.asarray(ct).ravel()[nz].astype(np.float64) / 255.0) / a,
        np.sqrt((l1 + 1e-3) / l2), 4.0 * np.sqrt(l1),
        np.hypot(my - CY, mxx - CX) / DISK_R,
        s(np.asarray(img).ravel()[nz].astype(np.float64) / 255.0) / a,
        s(np.asarray(flat).ravel()[nz].astype(np.float64) / 128.0) / a,
    ], 1).astype(np.float32)
    return feats, area

def overlap_matrices(lab, n, area, gt):
    L = lab.ravel()
    out = []
    for rec in gt["records"]:
        inter = np.zeros((len(rec["idx"]), n), np.float64)
        if n:
            for j, idx in enumerate(rec["idx"]):
                inter[j] = np.bincount(L[idx], minlength=n + 1)[1:]
        union = rec["area"][:, None] + area[None, :] - inter
        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        out.append((inter, iou))
    return out

def grid_keep(feats, area, flt):
    th, mc, ma = flt
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(th)) & (feats[:, F_IDX["mean_p"]] >= mc - 1e-9)
            & (area >= ma))

def prefilter_keep(feats, area):
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(CFG.scorer_prefilter_max))
            & (area >= CFG.scorer_prefilter_area))

def load_maps(fg, ct):
    if isinstance(fg, (str, Path)):
        fg = np.load(fg, mmap_mode="r")
    if isinstance(ct, (str, Path)):
        ct = np.load(ct, mmap_mode="r")
    return np.asarray(fg), np.asarray(ct)

# ---- process-pool workers (fork: they see GT_INDEX, CACHE, CFG of the parent at fork time) ----
def score_image_grid(task):
    name, fg, ct, geoms, filters = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    gt = GT_INDEX[name]
    out = np.zeros((len(geoms), len(filters), 5), np.float64)
    for gi, g in enumerate(geoms):
        lab, n = geometry_labels(fg, ct, g)
        feats, area = component_features(lab, n, fg, ct, img, flat)
        mats = overlap_matrices(lab, n, area, gt)
        for fi, flt in enumerate(filters):
            keep = grid_keep(feats, area, flt)
            for _, iou in mats:
                out[gi, fi] += match_stats(iou[:, keep])
    return name, out

def scorer_rows(task):
    name, fg, ct, geom = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n = geometry_labels(fg, ct, geom)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n, np.float64)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    pre = prefilter_keep(feats, area)
    return name, feats[pre], area[pre], y[pre], [iou[:, pre] for _, iou in mats]

_POOL = None
def get_pool(refresh=False):
    """Fork-based CPU pool. Refresh after defining new globals that workers must see."""
    global _POOL
    if refresh and _POOL is not None:
        _POOL.shutdown(wait=True)
        _POOL = None
    if _POOL is None:
        _POOL = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    return _POOL

def stats_table(total, geoms, filters):
    rows = []
    for gi, g in enumerate(geoms):
        for fi, f in enumerate(filters):
            rows.append({"t_low": g[0], "contact": g[1], "offset": g[2], "t_high": f[0], "min_mean": f[1],
                         "min_area": f[2], **summarize(total[gi, fi])})
    return pd.DataFrame(rows)

def pick_best(table):
    t = table.copy()
    t["_c"] = t["contact"].isna()
    t["_o"] = (t["offset"] == 0)
    t = t.sort_values(["pq", "_c", "_o", "min_area", "min_mean"], ascending=[False, False, False, True, True])
    return t.iloc[0].drop(["_c", "_o"]).to_dict()

def as_geom(row):
    c = row["contact"]
    return (float(row["t_low"]), None if (c is None or (isinstance(c, float) and np.isnan(c))) else float(c), int(row["offset"]))

def as_filter(row):
    return (float(row["t_high"]), float(row["min_mean"]), int(row["min_area"]))

print("PQ engine ready. Features:", FEATURES)


## Cell 09 — V4 detections (read from Drive; no detector is run)


In [ ]:
DET_DIR = DET_RUN / "detections"
DET_FOLDS = sorted(int(p.parent.name.replace("fold", "")) for p in (DET_RUN / "detector").glob("fold*/done.json"))
if not DET_FOLDS:
    raise FileNotFoundError(f"No detector done.json under {DET_RUN / 'detector'}")

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

DET_OOF, _test_by_fold = {}, {}
for fold in DET_FOLDS:
    sha = json.loads((DET_RUN / "detector" / f"fold{fold}" / "done.json").read_text())["weights_sha256"][:8]
    DET_OOF.update(load_pickle(DET_DIR / f"oof_fold{fold}_{sha}.pkl"))
    _test_by_fold[fold] = load_pickle(DET_DIR / f"test_fold{fold}_{sha}.pkl")
DET_TEST = {n: [_test_by_fold[f][n] for f in DET_FOLDS] for n in sorted(TEST_MAPS)}
print("Detections from:", DET_RUN_NAME, "| detector folds:", DET_FOLDS, "| out-of-fold images with detections:", len(DET_OOF),
      "| test images:", len(DET_TEST))


## Cell 10 — Candidates per variant: V4 fusion, plus the recall probe and the Otsu re-snap (with self-tests)
A variant is `{t_low, prefilter, mask, offset}`. `base` = V4's `v4_full` candidates exactly. The re-snap works on the UNet
candidates only, before detector-only candidates are added, so every candidate stays one disjoint region.


In [ ]:
DET_FEATURES = ("det_score", "det_iou", "det_cov")
ALL_FEATURES = tuple(FEATURES) + DET_FEATURES + ("src",)
N_V3F, N_DETF = len(FEATURES), len(DET_FEATURES)
_GROW_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG.det_grow + 1, 2 * CFG.det_grow + 1))

def rasterize_detections(dets):
    """Detections of one detector -> list of (score, y0, x0, boolean mask of its bounding window), best first."""
    out = []
    for s, p in zip(dets["scores"], dets["polys"]):
        if p is None or len(p) < 3:
            continue
        q = np.rint(np.asarray(p, np.float64)).astype(np.int32)
        x0, y0 = max(int(q[:, 0].min()), 0), max(int(q[:, 1].min()), 0)
        x1, y1 = min(int(q[:, 0].max()) + 1, W), min(int(q[:, 1].max()) + 1, H)
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        roi = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(roi, [q - np.array([x0, y0], np.int32)], 1)
        if int(roi.sum()) < CFG.det_min_area:
            continue
        out.append((float(s), y0, x0, roi.astype(bool)))
    out.sort(key=lambda t: -t[0])
    return out

def detection_evidence(lab, n, area, rast):
    """Per candidate: score of the detection sharing most pixels with it, IoU with that detection, fraction covered."""
    score, iou, cover = np.zeros(n), np.zeros(n), np.zeros(n)
    if n == 0 or not rast:
        return score, iou, cover
    best = np.zeros(n)
    covered = np.zeros(lab.shape, bool)
    for s, y0, x0, m in rast:
        h, w = m.shape
        covered[y0:y0 + h, x0:x0 + w] |= m
        inter = np.bincount(lab[y0:y0 + h, x0:x0 + w][m], minlength=n + 1)[1:].astype(np.float64)
        a = float(m.sum())
        upd = (inter >= CFG.det_link_frac * np.minimum(area, a)) & (inter > best)
        if upd.any():
            best[upd] = inter[upd]
            score[upd] = s
            iou[upd] = inter[upd] / (area[upd] + a - inter[upd])
    cover = np.bincount(lab[covered], minlength=n + 1)[1:] / np.maximum(area, 1.0)
    return score, iou, cover

def largest_piece(mask):
    lab, n = ndi.label(mask, structure=CONN)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    return lab == int(sizes.argmax())

def consensus_scores(rasts):
    """Every detection gets the mean, over all detectors, of the best score among that detector's detections
    that overlap it (IoU >= 0.3). With one detector this is its own score; with four, a detection found by only
    one of them is worth a quarter. Keeps test images (4 detectors) comparable to out-of-fold images (1 detector)."""
    items = [(d, s, y0, x0, m, float(m.sum())) for d, rast in enumerate(rasts) for s, y0, x0, m in rast]
    out = []
    for d, s, y0, x0, m, a in items:
        total = s
        for d2 in range(len(rasts)):
            if d2 == d:
                continue
            best = 0.0
            for s2, v0, u0, m2 in rasts[d2]:
                if s2 <= best:
                    break                                         # sorted by score: nothing better follows
                ya, yb = max(y0, v0), min(y0 + m.shape[0], v0 + m2.shape[0])
                xa, xb = max(x0, u0), min(x0 + m.shape[1], u0 + m2.shape[1])
                if ya >= yb or xa >= xb:
                    continue
                inter = float((m[ya - y0:yb - y0, xa - x0:xb - x0] & m2[ya - v0:yb - v0, xa - u0:xb - u0]).sum())
                if inter >= 0.3 * (a + float(m2.sum()) - inter):
                    best = s2
            total += best
        out.append((total / max(len(rasts), 1), y0, x0, m))
    out.sort(key=lambda t: -t[0])
    return out

def detector_only_candidates(lab, fg, rasts):
    """Objects the detectors found where the UNet has no candidate. The mask is the UNet's probability inside the
    (slightly grown) detection when there is enough of it, otherwise the detection itself. All disjoint."""
    pooled = consensus_scores(rasts)
    occupied = lab > 0
    new = np.zeros(lab.shape, np.int32)
    kinds, g = [], CFG.det_grow
    for s, y0, x0, m in pooled:
        if s < CFG.det_only_floor:
            break
        h, w = m.shape
        if int((m & occupied[y0:y0 + h, x0:x0 + w]).sum()) > CFG.det_only_max_claimed * int(m.sum()):
            continue                                              # already represented
        ya, xa, yb, xb = max(y0 - g, 0), max(x0 - g, 0), min(y0 + h + g, H), min(x0 + w + g, W)
        big = np.zeros((yb - ya, xb - xa), np.uint8)
        big[y0 - ya:y0 - ya + h, x0 - xa:x0 - xa + w] = m
        free = ~occupied[ya:yb, xa:xb]
        det_free = (big > 0) & free
        claim = (cv2.dilate(big, _GROW_KERNEL) > 0) & free & (fg[ya:yb, xa:xb] >= qt(CFG.det_claim_t))
        if int(claim.sum()) >= CFG.det_claim_min_frac * int(det_free.sum()):
            mask, kind = largest_piece(claim), 1                  # drawn by the UNet
        else:
            mask, kind = largest_piece(det_free), 2               # drawn by the detector
        if int(mask.sum()) < CFG.det_only_min_area:
            continue
        kinds.append(kind)
        new[ya:yb, xa:xb][mask] = len(kinds)
        occupied[ya:yb, xa:xb] |= mask
    return new, np.asarray(kinds, np.float32)


BASE_VAR = {"t_low": None, "pre": True, "mask": "none", "offset": None}

def var_geom(var):
    return (GEOM[0] if var.get("t_low") is None else float(var["t_low"]), GEOM[1],
            GEOM[2] if var.get("offset") is None else int(var["offset"]))

def prefilter_var(feats, area, var):
    if var.get("pre", True):
        return prefilter_keep(feats, area)
    return area >= CFG.probe_prefilter_area

_CLOSE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (CFG.resnap_close, CFG.resnap_close))
_SNAP_GROW_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG.resnap_grow + 1, 2 * CFG.resnap_grow + 1))

def snap_one(pred, flat_win, disk_win, mode):
    """Re-draw one candidate on the dark pixels of its window (Otsu split of the flattened image)."""
    vals = flat_win[disk_win]
    if vals.size < CFG.resnap_min_pixels or vals.min() == vals.max():
        return pred
    thr, _ = cv2.threshold(np.ascontiguousarray(vals.reshape(-1, 1)), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark = (flat_win <= thr) & disk_win
    near = cv2.dilate(pred.astype(np.uint8), _SNAP_GROW_KERNEL) > 0
    cand = cv2.morphologyEx((dark & near).astype(np.uint8), cv2.MORPH_CLOSE, _CLOSE_KERNEL) > 0
    cand &= near
    cl, cn = ndi.label(cand, structure=STRUCT8)
    if cn == 0:
        return pred
    touch = np.unique(cl[pred & (cl > 0)])
    snapped = ndi.binary_fill_holes(np.isin(cl, touch[touch > 0]))
    new = (pred | snapped) if mode == "grow" else snapped
    ratio = new.sum() / max(pred.sum(), 1)
    if not (CFG.resnap_area_lo <= ratio <= CFG.resnap_area_hi) or not new.any():
        return pred
    return new

def resnap_labels(lab, n, fg, flat, mode):
    """Re-snap every candidate, most confident first; a pixel already taken is never given to another candidate."""
    if n == 0:
        return lab, n
    order = np.argsort(-np.asarray(ndi.maximum(fg, lab, index=np.arange(1, n + 1))), kind="stable")
    objs = ndi.find_objects(lab)
    out = np.zeros(lab.shape, np.int32)
    taken = np.zeros(lab.shape, bool)
    m = CFG.resnap_margin
    for k in order:
        sl = objs[k]
        if sl is None:
            continue
        win = (slice(max(sl[0].start - m, 0), min(sl[0].stop + m, H)), slice(max(sl[1].start - m, 0), min(sl[1].stop + m, W)))
        pred = lab[win] == k + 1
        new = snap_one(pred, np.asarray(flat[win]), DISK[win], mode) & ~taken[win]
        if not new.any():
            new = pred & ~taken[win]
        out[win][new] = k + 1
        taken[win] |= new
    return relabel(out)

def build_candidates(fg, ct, img, flat, det_list, var=BASE_VAR, with_detector_only=True):
    """V3 candidates under the variant's geometry/prefilter (optionally re-snapped) + V4 detector-only candidates.
    Returns (label map, n candidates, n from V3, feature matrix [n, len(ALL_FEATURES)], areas)."""
    lab0, n0 = geometry_labels(fg, ct, var_geom(var))
    feats0, area0 = component_features(lab0, n0, fg, ct, img, flat)
    pre = prefilter_var(feats0, area0, var)
    lut = np.zeros(n0 + 1, np.int32)
    ids = np.flatnonzero(pre) + 1
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    lab, n_v3 = lut[lab0], len(ids)
    if var.get("mask", "none") != "none" and n_v3:
        lab, n_v3 = resnap_labels(lab, n_v3, fg, flat, var["mask"])
    rasts = [rasterize_detections(d) for d in det_list]
    kinds = np.zeros(0, np.float32)
    if with_detector_only and any(rasts):
        new, kinds = detector_only_candidates(lab, fg, rasts)
        if len(kinds):
            lab = np.where(new > 0, new + n_v3, lab)
    n = n_v3 + len(kinds)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    ev = np.zeros((n, N_DETF))
    for rast in rasts:
        ev += np.stack(detection_evidence(lab, n, area, rast), 1)
    if rasts:
        ev /= len(rasts)
    src = np.r_[np.zeros(n_v3, np.float32), kinds]
    X = np.concatenate([feats.astype(np.float32), ev.astype(np.float32), src[:, None].astype(np.float32)], 1)
    return lab, n, n_v3, X, area

def fusion_tests():
    """V4's fusion self-test (same synthetic image, fixed geometry) plus the new re-snap and prefilter behaviour."""
    T = {"t_low": 0.4, "pre": True, "mask": "none", "offset": 0}
    saved_contact = GEOM
    globals()["GEOM"] = (0.4, None, 0)
    try:
        fg = np.zeros((H, W), np.uint8); ct = np.zeros((H, W), np.uint8)
        img = np.full((H, W), 120, np.uint8); flat = np.full((H, W), 128, np.uint8)
        fg[100:110, 100:300] = 230
        fg[200:210, 100:300] = 220
        fg[300:308, 100:260] = 70
        box = lambda y0, y1, x0, x1: np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], np.float32)
        dets = {"scores": np.array([0.9, 0.6, 0.4, 0.03], np.float32),
                "polys": [box(99, 110, 98, 301), box(299, 308, 98, 262), box(400, 409, 100, 250), box(450, 459, 100, 250)]}
        lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets], T)
        assert n_v3 == 2 and n == 4, (n_v3, n)
        col = {k: i for i, k in enumerate(ALL_FEATURES)}
        a, b = lab[105, 200] - 1, lab[205, 200] - 1
        assert X[a, col["det_score"]] > 0.85 and X[a, col["det_iou"]] > 0.7 and X[a, col["det_cov"]] > 0.95
        assert X[b, col["det_score"]] == 0 and X[b, col["det_cov"]] == 0
        c, d = lab[304, 150] - 1, lab[404, 150] - 1
        assert X[c, col["src"]] == 1 and X[d, col["src"]] == 2 and lab[454, 150] == 0
        # prefilter off + lower t_low turns the weak strip C (p = 0.27) into a UNet candidate
        lab2, n2, n2v, X2, _ = build_candidates(fg, ct, img, flat, [dets], {**T, "t_low": 0.25, "pre": False})
        assert n2v == 3 and X2[lab2[304, 150] - 1, col["src"]] == 0, (n2v,)
        # re-snap: a dark filament 12 px wide, predicted 6 px wide -> snap/grow recover it; bright pixels are never added
        fg3 = np.zeros((H, W), np.uint8); flat3 = flat.copy()
        flat3[1000:1012, 900:1100] = 60
        fg3[1002:1010, 905:1095] = 230
        base_area = build_candidates(fg3, ct, img, flat3, [], T)[4][0]
        for mode in ("snap", "grow"):
            l3, n3, _, _, a3 = build_candidates(fg3, ct, img, flat3, [], {**T, "mask": mode})
            assert n3 == 1 and a3[0] > 1.3 * base_area and l3[1000, 1000] and not l3[995, 1000], (mode, a3, base_area)
        # disjointness after re-snap: two touching dark strips stay two disjoint candidates
        fg4 = np.zeros((H, W), np.uint8); flat4 = flat.copy()
        flat4[1500:1530, 500:700] = 60
        fg4[1502:1512, 505:695] = 230; fg4[1518:1528, 505:695] = 200
        l4, n4, _, _, _ = build_candidates(fg4, ct, img, flat4, [], {**T, "mask": "grow"})
        assert n4 == 2 and all((l4 == k).any() for k in (1, 2))
    finally:
        globals()["GEOM"] = saved_contact
    print("PASS: V4 fusion, recall-probe prefilter, Otsu re-snap (snap/grow), disjointness")

fusion_tests()


## Cell 11 — Run a variant: out-of-fold candidates, cross-fitted scorer (V4 `v4_full` setup), every threshold
The scorer is V4's gradient-boosted regressor on all candidates with all features, weighted by the number of annotator sets.
Per variant we keep the per-image statistics at every threshold, which the nested audit needs.


In [ ]:
EVAL_NAMES = sorted(n for n in OOF_PATHS if n in DET_OOF)
IMG_FOLD = np.array([FOLD_OF[n] for n in EVAL_NAMES])
FOLDS_HERE = sorted(np.unique(IMG_FOLD))
if len(FOLDS_HERE) < 2:
    raise RuntimeError("Step 0 needs at least two folds of out-of-fold maps.")
TAUS = np.array(CFG.scorer_taus)

def oof_task(task):
    name, fgp, ctp, dets, var = task
    fg, ct = load_maps(fgp, ctp)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets], var)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    return name, X, area, y, len(mats), [iou for _, iou in mats]

def make_hgb():
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=40, l2_regularization=1.0, random_state=CFG.seed)

def image_stats(u, off, ious, tau):
    out = np.zeros((len(ious), 5))
    for i in range(len(ious)):
        keep = u[off[i]:off[i + 1]] >= tau
        for iou in ious[i]:
            out[i] += match_stats(iou[:, keep])
    return out

def pq_value(st):
    den = st[1] + 0.5 * (st[2] + st[3])
    return float(st[0] / den) if den else 0.0

def paired_bootstrap(a, b, n=None):
    n = CFG.bootstrap if n is None else n
    rng = np.random.default_rng(CFG.seed)
    w = rng.multinomial(len(a), np.full(len(a), 1.0 / len(a)), size=n).astype(np.float64)
    sa, sb = w @ a[:, :4], w @ b[:, :4]
    pq = lambda s: s[:, 0] / np.maximum(s[:, 1] + 0.5 * (s[:, 2] + s[:, 3]), 1e-9)
    return np.percentile(pq(sa) - pq(sb), [2.5, 97.5])

VAR_CACHE = LOCAL_ROOT / "variants"
VAR_CACHE.mkdir(parents=True, exist_ok=True)
RESULTS = {}

def run_variant(vname, var):
    """Candidates for every out-of-fold image -> cross-fitted u -> per-image stats at every threshold -> final scorer."""
    t = time.time()
    key = object_hash({"impl": "v5s0-1", "var": var, "names": EVAL_NAMES, "geom": var_geom(var), "cfg": asdict(CFG)})[:12]
    cache = VAR_CACHE / f"{vname}_{key}.pkl"
    if cache.is_file():
        RESULTS[vname] = load_pickle(cache)
        print(f"{vname}: reused cached result")
        return RESULTS[vname]
    pool = get_pool()
    rows = list(tqdm(pool.map(oof_task, [(n, *OOF_PATHS[n], DET_OOF[n], var) for n in EVAL_NAMES], chunksize=2),
                     total=len(EVAL_NAMES), desc=f"{vname}: candidates"))
    assert [r[0] for r in rows] == EVAL_NAMES
    X = np.concatenate([r[1] for r in rows]).astype(np.float32)
    Y = np.concatenate([r[3] for r in rows])
    counts = [len(r[3]) for r in rows]
    off = np.r_[0, np.cumsum(counts)]
    img_of = np.repeat(np.arange(len(rows)), counts)
    K = np.repeat(np.array([r[4] for r in rows], np.float64), counts)
    ious = [r[5] for r in rows]
    del rows
    group = IMG_FOLD[img_of]
    u = np.full(len(Y), -np.inf)
    for g in FOLDS_HERE:
        tr, te = group != g, group == g
        if tr.sum() < (40 if CFG.debug_run else 200) or te.sum() == 0:
            u[te] = 0.0
            continue
        u[te] = make_hgb().fit(X[tr], Y[tr], sample_weight=K[tr]).predict(X[te])
    stats_all = np.stack([image_stats(u, off, ious, tau) for tau in TAUS])           # [taus, images, 5]
    pqs = np.array([pq_value(s.sum(0)) for s in stats_all])
    best = int(np.lexsort((TAUS, -pqs))[0])                                        # best PQ, then smallest tau
    final = make_hgb().fit(X, Y, sample_weight=K)
    res = {"var": var, "u": u, "off": off, "stats_all": stats_all, "tau": float(TAUS[best]), "stats": stats_all[best],
           "model": final, "n_cand": len(Y), "n_v3": int((X[:, -1] == 0).sum()), "hit_rate": float((Y > 0).mean()),
           "minutes": (time.time() - t) / 60}
    save_pickle(cache, res)
    RESULTS[vname] = res
    s = summarize(res["stats"].sum(0))
    print(f"{vname}: PQ {s['pq']:.4f} (SQ {s['sq']:.3f} RQ {s['rq']:.3f} P {s['precision']:.3f} R {s['recall']:.3f}) "
          f"tau {res['tau']:g} | {res['n_cand']} candidates | {res['minutes']:.1f} min")
    return res

def compare(names, baseline="base"):
    base = RESULTS[baseline]["stats"]
    rows = []
    for v in names:
        r = RESULTS[v]
        s = summarize(r["stats"].sum(0))
        fold = {g: pq_value(r["stats"][IMG_FOLD == g].sum(0)) for g in FOLDS_HERE}
        bfold = {g: pq_value(base[IMG_FOLD == g].sum(0)) for g in FOLDS_HERE}
        lo, hi = paired_bootstrap(r["stats"], base) if v != baseline else (0.0, 0.0)
        rows.append({"variant": v, **{k: r["var"].get(k) for k in ("t_low", "pre", "mask", "offset")}, "tau": r["tau"],
                     "pq": s["pq"], "sq": s["sq"], "rq": s["rq"], "precision": s["precision"], "recall": s["recall"],
                     "tp": s["tp"], "fp": s["fp"], "fn": s["fn"], "candidates": r["n_cand"],
                     "gain": s["pq"] - pq_value(base.sum(0)), "ci_low": float(lo), "ci_high": float(hi),
                     "fold_wins": int(sum(fold[g] > bfold[g] + 1e-12 for g in FOLDS_HERE)), "folds": len(FOLDS_HERE)})
    return pd.DataFrame(rows)

def adopted(table, baseline="base"):
    need = min(CFG.min_fold_wins, len(FOLDS_HERE))
    ok = table[(table.variant != baseline) & (table.gain >= CFG.min_gain) & (table.fold_wins >= need)]
    return ok.sort_values("pq", ascending=False).variant.tolist()

get_pool(refresh=True)                          # workers must see the functions above
print("Images:", len(EVAL_NAMES), "| folds:", FOLDS_HERE, "| thresholds:", len(TAUS))


## Cell 12 — Base (must reproduce V4), recall probe, Otsu re-snap


In [ ]:
VARIANTS = {"base": dict(BASE_VAR)}
if os.environ.get("SFSC_VARIANT_SET", "all") == "all":
    for t in CFG.probe_t_lows:
        VARIANTS[f"tlow{t:g}"] = {**BASE_VAR, "t_low": t, "pre": False}
    VARIANTS["snap"] = {**BASE_VAR, "mask": "snap"}
    VARIANTS["grow"] = {**BASE_VAR, "mask": "grow"}

for vname, var in VARIANTS.items():
    run_variant(vname, var)

_base_pq = pq_value(RESULTS["base"]["stats"].sum(0))
if not CFG.debug_run and len(EVAL_NAMES) == len(UNIQUE_FILES) and DET_RUN_NAME == "sfsc_fusion_v4":
    print(f"\nBase vs V4 manifest: {_base_pq:.4f} vs {CFG.v4_reference_pq:.4f}")
    if abs(_base_pq - CFG.v4_reference_pq) > 0.002:
        warnings.warn("The base variant does not reproduce V4; comparisons are still against this run's base.")

print(f"Base with detections from {DET_RUN_NAME}: {_base_pq:.4f} | V4 (V4 detector): {CFG.v4_reference_pq:.4f}"
      f" | difference {_base_pq - CFG.v4_reference_pq:+.4f}")
STAGE1 = compare(list(VARIANTS))
print("\nCross-fitted out-of-fold results (gain, interval and fold wins are against base):")
print(STAGE1.round(4).to_string(index=False))
STAGE1.to_csv(RUN / "stage1_variants.csv", index=False)
push_outputs()


## Cell 13 — Combination, final system, and the offset-0 probe
If a recall variant and a re-snap variant both pass, their combination is tested. The final system is the best passing variant
(base if none passes). Its boundary-offset-0 twin is built for the leaderboard probe only.


In [ ]:
passed = adopted(STAGE1)
best_recall = next((v for v in passed if v.startswith("tlow")), None)
best_mask = next((v for v in passed if v in ("snap", "grow")), None)
if best_recall and best_mask:
    VARIANTS["combo"] = {**VARIANTS[best_recall], "mask": VARIANTS[best_mask]["mask"]}
    run_variant("combo", VARIANTS["combo"])

ALL_TABLE = compare(list(VARIANTS))
FINAL_NAME = (adopted(ALL_TABLE) or ["base"])[0]
FINAL_VAR = VARIANTS[FINAL_NAME]
VARIANTS["final_offset0"] = {**FINAL_VAR, "offset": 0}
run_variant("final_offset0", VARIANTS["final_offset0"])

ALL_TABLE = compare(list(VARIANTS))
ALL_TABLE.to_csv(RUN / "variants_summary.csv", index=False)
print(ALL_TABLE.round(4).to_string(index=False))
print(f"\nFinal Step-0 system: {FINAL_NAME} {FINAL_VAR} | out-of-fold PQ "
      f"{pq_value(RESULTS[FINAL_NAME]['stats'].sum(0)):.4f} at tau {RESULTS[FINAL_NAME]['tau']:g}")
push_outputs()


## Cell 14 — Nested-selection audit: how optimistic is a number chosen and scored on the same folds?
For each of the 6 ways to split the 4 folds into 2 + 2: choose on folds A, score on folds B, and compare with the choice made on
all folds scored on the same B. (a) V3's grid recipe from `calibration_v3_per_image.npz`; (b) the threshold of the base scorer;
(c) this notebook's variant + threshold choice.


In [ ]:
from itertools import combinations

def nested(stat_of_choice, n_choices, img_fold):
    """stat_of_choice(c) -> [images, 5]. Returns mean PQ on B of (choice made on A) and of (choice made on all folds)."""
    folds = sorted(np.unique(img_fold))
    if len(folds) < 4:
        return None
    totals = [stat_of_choice(c) for c in range(n_choices)]
    glob = int(np.argmax([pq_value(t.sum(0)) for t in totals]))
    rows = []
    for A in combinations(folds, len(folds) // 2):
        a = np.isin(img_fold, A)
        pick = int(np.argmax([pq_value(t[a].sum(0)) for t in totals]))
        rows.append({"A": A, "nested_B": pq_value(totals[pick][~a].sum(0)), "global_B": pq_value(totals[glob][~a].sum(0)),
                     "pick": pick, "global": glob})
    df = pd.DataFrame(rows)
    return {"nested_B": float(df.nested_B.mean()), "global_B": float(df.global_B.mean()),
            "optimism": float(df.global_B.mean() - df.nested_B.mean()), "splits": df.to_dict("records")}

AUDIT = {}
_cal_json = json.loads((V3_RUN / "calibration_v3.json").read_text())
with np.load(V3_RUN / "calibration_v3_per_image.npz") as z:
    _cal_names, _cal_stats = z["names"].tolist(), z["stats"].astype(np.float64)        # [images, geoms, filters, 5]
_cal_fold = np.array([FOLD_OF[n] for n in _cal_names])
_ng, _nf = _cal_stats.shape[1], _cal_stats.shape[2]
AUDIT["v3_grid"] = nested(lambda c: _cal_stats[:, c // _nf, c % _nf], _ng * _nf, _cal_fold)
AUDIT["base_tau"] = nested(lambda c: RESULTS["base"]["stats_all"][c], len(TAUS), IMG_FOLD)
_choices = [(v, ti) for v in VARIANTS for ti in range(len(TAUS)) if v != "final_offset0"]
AUDIT["step0_variant_and_tau"] = nested(lambda c: RESULTS[_choices[c][0]]["stats_all"][_choices[c][1]], len(_choices), IMG_FOLD)
for k, v in AUDIT.items():
    if v:
        print(f"{k:24s} chosen on 2 folds -> PQ {v['nested_B']:.4f} on the other 2 | chosen on all -> {v['global_B']:.4f}"
              f" | optimism {v['optimism']:+.4f}")
write_json(RUN / "audit.json", AUDIT)


## Cell 15 — Test exports: final system, offset-0 probe, loose-threshold probe (audited; no image left empty)
The final scorers are fitted on every out-of-fold candidate of their variant and used only on test images.


In [ ]:
EXPORT = RUN / "test_export"
EXPORT.mkdir(parents=True, exist_ok=True)
TEST_NAMES = sorted(TEST_MAPS)

def export_task(task):
    name, var = task
    fg, ct = load_maps(*TEST_MAPS[name])
    img, flat = cached_arrays(cache_key("test", name), False)
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, DET_TEST[name], var)
    rles = [mask_api.encode(np.asfortranarray((lab == i).astype(np.uint8)))["counts"].decode("ascii")
            for i in range(1, n + 1)]
    return name, X, rles

def audit_csv(path, names):
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    if list(table.columns) != ["filament_id", "segmentation_rle"] or table.filament_id.duplicated().any():
        raise ValueError(f"{path.name}: wrong columns or duplicate filament ids")
    stems = {Path(n).stem for n in names}
    by_stem = defaultdict(list)
    for fid, rle in table.itertuples(index=False):
        stem, suffix = fid.rsplit("_", 1)
        by_stem[stem].append((int(suffix), rle))
    if not set(by_stem) <= stems:
        raise ValueError(f"{path.name}: rows for unknown images")
    for stem, items in by_stem.items():
        if sorted(i for i, _ in items) != list(range(1, len(items) + 1)):
            raise AssertionError(f"{path.name}: ids of {stem} are not 1..n")
        seen = np.zeros((H, W), bool)
        for _, rle in items:
            m = decode_mask({"size": [H, W], "counts": rle})
            if not m.any() or (seen & m).any():
                raise AssertionError(f"{path.name}: empty or overlapping mask in {stem}")
            seen |= m
    return len(table), len(stems) - len(by_stem)


def test_candidates(vname):
    pool = get_pool()                           # refreshed below, after export_task is defined
    out = list(tqdm(pool.map(export_task, [(n, VARIANTS[vname]) for n in TEST_NAMES], chunksize=1),
                    total=len(TEST_NAMES), desc=f"Test candidates ({vname})"))
    model = RESULTS[vname]["model"]
    return {name: (model.predict(X) if len(X) else np.zeros(0), rles) for name, X, rles in out}

def write_submission(cands, tau, label, never_empty=CFG.never_empty):
    rows, filled = [], 0
    for name in TEST_NAMES:
        u, rles = cands[name]
        keep = np.flatnonzero(u >= tau)
        if never_empty and len(keep) == 0 and len(u):
            keep, filled = np.array([int(np.argmax(u))]), filled + 1
        stem = Path(name).stem
        rows += [(f"{stem}_{j}", rles[i]) for j, i in enumerate(keep, 1)]
    path = EXPORT / f"submission_{label}.csv"
    pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"]).to_csv(path, index=False)
    n_rows, n_empty = audit_csv(path, TEST_NAMES)
    print(f"{label:28s}: {n_rows} filaments, {n_rows / len(TEST_NAMES):.2f} per image, {n_empty} empty images,"
          f" {filled} filled by the never-empty rule, tau {tau:.4f} - audit passed")
    return {"path": str(path), "rows": n_rows, "empty": n_empty, "filled": filled, "tau": float(tau)}

get_pool(refresh=True)                          # workers must see export_task and the test detections
EXPORTS = {}
if CFG.export_test:
    CANDS = {"base": test_candidates("base")}
    EXPORTS["v4_replica_check"] = write_submission(CANDS["base"], RESULTS["base"]["tau"], "v4_replica_check", never_empty=False)
    print("   (V4 exported 1077 filaments; the replica above should match when nothing upstream changed)")
    if FINAL_NAME not in CANDS:
        CANDS[FINAL_NAME] = test_candidates(FINAL_NAME)
    EXPORTS["a_final"] = write_submission(CANDS[FINAL_NAME], RESULTS[FINAL_NAME]["tau"], f"a_final_{FINAL_NAME}")
    CANDS["final_offset0"] = test_candidates("final_offset0")
    EXPORTS["b_offset0"] = write_submission(CANDS["final_offset0"], RESULTS["final_offset0"]["tau"], "b_offset0")
    _u_all = np.sort(np.concatenate([u for u, _ in CANDS[FINAL_NAME].values()]))[::-1]
    _k = int(round(CFG.loose_keep_per_image * len(TEST_NAMES)))
    _tau_loose = float(_u_all[min(_k, len(_u_all)) - 1]) if len(_u_all) else 0.0
    EXPORTS["c_loose"] = write_submission(CANDS[FINAL_NAME], _tau_loose, "c_loose_tau")
    for _keep in CFG.strict_keep_per_image:
        _k = int(round(_keep * len(TEST_NAMES)))
        _tau_strict = float(_u_all[min(_k, len(_u_all)) - 1]) if len(_u_all) else 0.0
        EXPORTS[f"d_strict{_keep:g}"] = write_submission(CANDS[FINAL_NAME], _tau_strict, f"d_strict{_keep:g}_tau")
    _oof_kept = sum((RESULTS[FINAL_NAME]["u"] >= RESULTS[FINAL_NAME]["tau"])) / len(EVAL_NAMES)
    print(f"Out-of-fold kept per image for {FINAL_NAME}: {_oof_kept:.2f}")
push_outputs()


## Cell 16 — Manifest


In [ ]:
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(RUN / "environment_step0.txt").write_text(freeze.stdout)
_fin = ALL_TABLE.set_index("variant").loc[FINAL_NAME]
manifest = {"algorithm": "V5 Step 0: V4 fusion + recall probe + Otsu re-snap, cross-fitted, nested audit",
            "final": FINAL_NAME, "final_variant": FINAL_VAR,
            "oof": {k: (float(_fin[k]) if k not in ("tp", "fp", "fn") else int(_fin[k]))
                    for k in ("tau", "pq", "sq", "rq", "precision", "recall", "tp", "fp", "fn", "gain", "ci_low", "ci_high")},
            "base_pq": _base_pq, "v4_reference_pq": CFG.v4_reference_pq,
            "variants": ALL_TABLE.to_dict("records"),
            "audit": {k: (None if v is None else {kk: v[kk] for kk in ("nested_B", "global_B", "optimism")})
                      for k, v in AUDIT.items()},
            "exports": EXPORTS, "config": asdict(CFG), "images_scored": len(EVAL_NAMES),
            "v3_run": str(V3_RUN), "v4_run": str(V4_RUN), "geometry_v3": GEOM,
            "python": platform.python_version(), "minutes": (time.time() - T0) / 60}
write_json(RUN / "manifest_step0.json", manifest)
push_outputs()
print(f"Final: {FINAL_NAME} | out-of-fold PQ {_fin.pq:.4f} (gain over base {_fin.gain:+.4f}, "
      f"95% interval {_fin.ci_low:+.4f} to {_fin.ci_high:+.4f}, fold wins {int(_fin.fold_wins)}/{int(_fin.folds)})")
print("Artifacts:", RUN, "| minutes:", round((time.time() - T0) / 60, 1))
if _POOL is not None:                            # a live worker pool keeps the kernel from exiting
    _POOL.shutdown(wait=True)
